In [29]:
# tests.ipynb
import requests
import json

BASE_URL = "http://localhost:8000/api/v1"

print("=== Test 1: Health Check ===")
response = requests.get(f"{BASE_URL}/health")
print(f"Status: {response.status_code}")
try:
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
except:
    print(f"Response text: {response.text}")

print("\n=== Test 2: Config ===")
response = requests.get(f"{BASE_URL}/config")
print(f"Status: {response.status_code}")
try:
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
except:
    print(f"Response text: {response.text}")


=== Test 1: Health Check ===
Status: 200
{
  "status": "healthy",
  "version": "1.0.0"
}

=== Test 2: Config ===
Status: 200
{
  "project_name": "Tariff Analysis System",
  "version": "1.0.0",
  "database_configured": true,
  "llm_configured": true,
  "qdrant_configured": true
}


In [27]:
print("\n=== Test 3: Tariff Analysis ===")
payload = {"tnved_code": "8428 10"}
response = requests.post(f"{BASE_URL}/analyze/tariff", json=payload)
print(f"Status: {response.status_code}")
print(f"Headers: {response.headers}")
print(f"Raw Response Text:\n{response.text}")
try:
    print(json.dumps(response.json(), indent=2, ensure_ascii=False))
except Exception as e:
    print(f"JSON Parse Error: {e}")



=== Test 3: Tariff Analysis ===
Status: 200
Headers: {'date': 'Sat, 18 Oct 2025 18:46:22 GMT', 'server': 'uvicorn', 'content-length': '5360', 'content-type': 'application/json'}
Raw Response Text:
{"metadata":{"tnved_code":"8428 10","tnved_name":"Машины и устройства для подъема, перемещения, погрузки или разгрузки (например, лифты, эскалаторы, конвейеры, канатные дороги) прочие:лифты и подъемники скиповые","okpd_codes":[{"okpd_code":"28.22.16","okpd_name":"Лифты, скиповые подъемники, эскалаторы и движущиеся пешеходные дорожки"}],"product_name":"Лифты","current_tariff":"0%"},"product_info":{"tariff_rate":0.0,"wto_rate":0.05,"has_certification":"да"},"metrics":{"unfriendly_share":0.0,"unfriendly_growth":0.0,"top1_avg_price":0.0,"other_avg_price":0.02},"tariff_measures":{"measures":["Мера 1"],"reasoning":["Возможность повышения тарифа, производство >= потребления"]},"nontariff_measures":{"measures":["Мера 5"],"reasoning":["Применима мера сертификации"]},"dashboard_data":{"import_dynamics

In [26]:

print("\n=== Test 4: Full RAG Analysis ===")
payload = {"tnved_code": "8428 10"}
response = requests.post(f"{BASE_URL}/analyze/full", json=payload)
print(f"Status: {response.status_code}")
print(f"Raw Response Text:\n{response.text[:500]}")
try:
    result = response.json()
    print(json.dumps(result, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"JSON Parse Error: {e}")



=== Test 4: Full RAG Analysis ===
Status: 200
Raw Response Text:
{"algorithm_result":{"metadata":{"tnved_code":"8428 10","tnved_name":"Машины и устройства для подъема, перемещения, погрузки или разгрузки (например, лифты, эскалаторы, конвейеры, канатные дороги) прочие:лифты и подъемники скиповые","okpd_codes":[{"okpd_code":"28.22.16","okpd_name":"Лифты, скиповые подъемники, эскалаторы и движущиеся пешеходные дорожки"}],"product_name":"Лифты","current_tariff":"0%"},"product_info":{"tariff_rate":0.0,"wto_rate":0.05,"has_certification":"да"},"metrics":{"unfriend
{
  "algorithm_result": {
    "metadata": {
      "tnved_code": "8428 10",
      "tnved_name": "Машины и устройства для подъема, перемещения, погрузки или разгрузки (например, лифты, эскалаторы, конвейеры, канатные дороги) прочие:лифты и подъемники скиповые",
      "okpd_codes": [
        {
          "okpd_code": "28.22.16",
          "okpd_name": "Лифты, скиповые подъемники, эскалаторы и движущиеся пешеходные дорожки"
        }


In [25]:
payload = {"tnved_code": "8428 10"}
response = requests.post(f"{BASE_URL}/analyze/tariff", json=payload)

print(f"Status: {response.status_code}")
if response.status_code == 200:
    data = response.json()
    print(f"\nMetadata: {data['metadata']['product_name']}")
    print(f"Dashboard fields: {list(data.get('dashboard_data', {}).keys())}")
    print(f"Import dynamics years: {len(data['dashboard_data']['import_dynamics'])}")
    print(f"Geography entries: {len(data['dashboard_data']['import_geography'])}")
    print(f"Top 5 prices: {len(data['dashboard_data']['top5_contract_prices'])}")
else:
    print(f"Error: {response.text}")


Status: 200

Metadata: Лифты
Dashboard fields: ['import_dynamics', 'production_dynamics', 'consumption_dynamics', 'import_geography', 'top5_contract_prices']
Import dynamics years: 3
Geography entries: 41
Top 5 prices: 5


In [30]:
import requests
import json

def get_structure(obj):
    if isinstance(obj, dict):
        return {k: get_structure(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        if len(obj) > 0:
            return [get_structure(obj[0])]
        return []
    elif isinstance(obj, bool):
        return "bool"
    elif isinstance(obj, int):
        return "int"
    elif isinstance(obj, float):
        return "float"
    elif isinstance(obj, str):
        return "str"
    elif obj is None:
        return "null"
    else:
        return type(obj).__name__


print("\n=== Test 4: Full RAG Analysis ===")
payload = {"tnved_code": "8428 10"}
response = requests.post(f"{BASE_URL}/analyze/full", json=payload)
print(f"Status: {response.status_code}\n")

try:
    result = response.json()
    structure = get_structure(result)
    print(json.dumps(structure, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"Error: {e}")
    print(f"Raw Response:\n{response.text[:500]}")



=== Test 4: Full RAG Analysis ===
Status: 200

{
  "algorithm_result": {
    "metadata": {
      "tnved_code": "str",
      "tnved_name": "str",
      "okpd_codes": [
        {
          "okpd_code": "str",
          "okpd_name": "str"
        }
      ],
      "product_name": "str",
      "current_tariff": "str"
    },
    "product_info": {
      "tariff_rate": "float",
      "wto_rate": "float",
      "has_certification": "str"
    },
    "metrics": {
      "unfriendly_share": "float",
      "unfriendly_growth": "float",
      "top1_avg_price": "float",
      "other_avg_price": "float"
    },
    "tariff_measures": {
      "measures": [
        "str"
      ],
      "reasoning": [
        "str"
      ]
    },
    "nontariff_measures": {
      "measures": [
        "str"
      ],
      "reasoning": [
        "str"
      ]
    },
    "dashboard_data": {
      "import_dynamics": [
        {
          "year": "int",
          "value_mln_usd": "float",
          "weight_tons": "float"
    